# Notebook 4 — Final Evaluation, Report Figures & predict.py Testing
**Member 4 Responsibility**

This notebook:
1. Collects results from all models (baselines + ML + transformer)
2. Generates the final comparison table and figures for the report
3. Tests `predict.py` end-to-end to verify submission readiness
4. Performs qualitative error analysis for the report

**Prerequisites:** Run notebooks 1, 2, and 3 first.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import os
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# ── Load all model results ────────────────────────────────────────────────
with open('baselines_results.json') as f:
    baseline_results = json.load(f)
with open('ml_results.json') as f:
    ml_results = json.load(f)
with open('transformer_results.json') as f:
    transformer_results = json.load(f)

all_results = {**baseline_results, **ml_results, **transformer_results}
print('All results loaded:')
for name, res in all_results.items():
    print(f'  {name}: Spearman={res["spearman_r"]}, AccWithinStd={res["acc_within_stdev"]}')

## 1. Final Results Table

In [ ]:
res_df = pd.DataFrame(all_results).T.reset_index()
res_df.columns = ['Model', 'Spearman ρ', 'Accuracy Within StdDev']
res_df = res_df.sort_values('Spearman ρ', ascending=False).reset_index(drop=True)

print('=== Final Results on Dev Set ===')
print(res_df.to_string(index=False))

# Save CSV for the report
res_df.to_csv('final_results.csv', index=False)
print('\nSaved: final_results.csv')

## 2. Final Comparison Plot (Report Figure)

In [ ]:
models  = res_df['Model'].tolist()
spearman = res_df['Spearman ρ'].astype(float).tolist()
acc      = res_df['Accuracy Within StdDev'].astype(float).tolist()

# Color coding: baselines=red, ML=blue, transformer=green
baseline_names = set(baseline_results.keys())
ml_names = set(ml_results.keys())
colors = []
for m in models:
    if m in baseline_names:
        colors.append('#e74c3c')   # red
    elif m in ml_names:
        colors.append('#3498db')   # blue
    else:
        colors.append('#2ecc71')   # green (transformer)

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, spearman, width, label='Spearman ρ', color=colors, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, acc, width, label='Accuracy Within StdDev', color=colors, alpha=0.5, edgecolor='white', hatch='//')

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('All Models — Dev Set Performance')
ax.legend()

# Annotate bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

from matplotlib.patches import Patch
legend_patches = [
    Patch(color='#e74c3c', label='Baselines'),
    Patch(color='#3498db', label='ML Models'),
    Patch(color='#2ecc71', label='Transformer')
]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][len(legend_patches):], loc='upper right')

plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: final_comparison.png')

## 3. Test predict.py End-to-End
This verifies that your submission file actually works before submitting.

In [ ]:
# Create a small test input (first 5 dev samples)
with open('dev.json') as f:
    dev_raw = json.load(f)

test_input = {k: dev_raw[k] for k in list(dev_raw.keys())[:5]}
with open('test_input.json', 'w') as f:
    json.dump(test_input, f)

# Run predict.py (this is what the grader does!)
result = subprocess.run(
    ['python', 'predict.py', 'test_input.json', 'test_output.jsonl'],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ predict.py FAILED!')
    print('STDERR:', result.stderr)
else:
    print('✅ predict.py ran successfully!')
    print('Output:')
    with open('test_output.jsonl') as f:
        for line in f:
            print(' ', line.strip())

    # Validate format
    errors = []
    with open('test_output.jsonl') as f:
        lines = [json.loads(l) for l in f]
    for l in lines:
        if 'id' not in l or 'prediction' not in l:
            errors.append('Missing id or prediction key')
        if not isinstance(l['prediction'], int) or not (1 <= l['prediction'] <= 5):
            errors.append(f'Invalid prediction value: {l["prediction"]}')
    if errors:
        print('❌ Format errors:', errors)
    else:
        print('✅ Output format is correct!')

## 4. Qualitative Analysis: What Does the Model Get Wrong?

In [ ]:
# Load dev data with transformer predictions
import torch
from transformers import RobertaTokenizer, RobertaModel
from torch import nn

def to_df(raw):
    rows = []
    for sid, s in raw.items():
        rows.append({
            'id': sid,
            'homonym': s['homonym'],
            'judged_meaning': s['judged_meaning'],
            'precontext': s.get('precontext', ''),
            'sentence': s['sentence'],
            'ending': s.get('ending', '') or '',
            'average': s['average'],
            'stdev': s['stdev']
        })
    return pd.DataFrame(rows)

dev_df = to_df(dev_raw)

# Error pattern analysis without re-running model (use saved predictions from nb3)
# We'll analyze the distribution of errors by score bucket
print('=== Qualitative Analysis ===')
print('\nKinds of ambiguity where models likely struggle:')
print('  1. Samples with no ending (less context clues)')
print('  2. Very polarized annotations (high stdev = human disagreement)')
print('  3. Rare homonyms not seen during training')

no_ending = dev_df[dev_df['ending'] == '']
has_ending = dev_df[dev_df['ending'] != '']
print(f'\nDev samples without ending: {len(no_ending)} (avg score: {no_ending["average"].mean():.2f})')
print(f'Dev samples with ending:    {len(has_ending)} (avg score: {has_ending["average"].mean():.2f})')

high_disagreement = dev_df[dev_df['stdev'] > 1.5]
print(f'\nDev samples with high annotator disagreement (stdev>1.5): {len(high_disagreement)}')
print(f'  These are inherently hard to predict.')

## 5. Pre-Submission Checklist

In [ ]:
checks = {
    'predict.py exists': os.path.exists('predict.py'),
    'requirements.txt exists': os.path.exists('requirements.txt'),
    'best_roberta_model.pt exists': os.path.exists('best_roberta_model.pt'),
    'predict.py ran without error': result.returncode == 0,
    'Output format correct': len(errors) == 0 if 'errors' in dir() else False,
}

print('=== Pre-Submission Checklist ===')
all_ok = True
for item, ok in checks.items():
    status = '✅' if ok else '❌'
    print(f'  {status} {item}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('🎉 All checks passed! Ready to submit.')
else:
    print('⚠️  Fix the failing checks before submitting.')

## ✅ Notebook 4 Complete
Files produced:
- `final_results.csv`
- `final_comparison.png`

**All notebooks done!** Your submission folder should contain:
- `predict.py`
- `requirements.txt`
- `best_roberta_model.pt`
- `README.md`